In [ ]:
from textwrap import dedent

from pathlib import Path
import pandas as pd
import re
import sys


DATA_DIR = Path("uncleaned_data")

MITIGATIONS_XLSX = DATA_DIR / "ics-attack-v17.1-mitigations.xlsx"
TACTICS_XLSX     = DATA_DIR / "ics-attack-v17.1-tactics.xlsx"
TECHNIQUES_XLSX  = DATA_DIR / "ics-attack-v17.1-techniques.xlsx"
GROUPS_XLSX      = DATA_DIR / "ics-attack-v17.1-groups.xlsx"
ASSETS_XLSX      = DATA_DIR / "ics-attack-v17.1-assets.xlsx"

OUT_DIR = Path("cleaned data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sheet names (exact, case-insensitive)
SHEETS = {
    "mitigations": {
        "mitigations": "mitigations",
        "techniques_addressed": "techniques addressed",
    },
    "tactics": {
        "tactics": "tactics",
    },
    "techniques": {
        "techniques": "techniques",
        "targeted_assets": "targeted assets",
    },
    "groups": {
        "groups": "groups",
        "associated_software": "associated software",
        "techniques_used": "techniques used",
    },
    "assets": {
        "assets": "assets",
    },
}

# Output CSV paths
OUT = {
    "mitigations": OUT_DIR / "mitigations_node.csv",
    "mitigation_technique": OUT_DIR / "MITIGATES.csv",
    "tactics": OUT_DIR / "tactics_node.csv",
    "techniques": OUT_DIR / "techniques_node.csv",
    "technique_assets": OUT_DIR / "ATTACKS.csv",
    "groups": OUT_DIR / "groups_node.csv",
    "associated_software": OUT_DIR / " malware_node.csv",
    "group_technique": OUT_DIR / "USE_TECHNIQUE.csv",
    "assets": OUT_DIR / "asset_node.csv",
}

# =====================
# Helpers
# =====================
def die(msg: str) -> None:
    raise SystemExit(msg)

def excel_or_die(path: Path) -> None:
    if not path.exists():
        die(f"Excel file not found: {path.resolve()}")

def norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip()).lower()

def find_sheet_exact(sheet_names, desired_name: str):
    """Return actual sheet name that matches desired_name exactly (case-insensitive, whitespace-normalized)."""
    target = norm(desired_name)
    for s in sheet_names:
        if norm(s) == target:
            return s
    return None

def pick_col_exact(df: pd.DataFrame, desired_name: str):
    """Return actual column name that matches desired_name exactly (case-insensitive, whitespace-normalized)."""
    target = norm(desired_name)
    for c in df.columns:
        if norm(c) == target:
            return c
    return None

def require_cols(df: pd.DataFrame, sheet_label: str, required_map):
    """
    required_map: list of tuples (label_for_error, [candidate_col_names...])
    Returns dict of {label_for_error: actual_col_name}
    Tries each candidate in order, using exact/normalized match.
    """
    resolved = {}
    missing = []
    for label, candidates in required_map:
        actual = None
        for cand in candidates:
            actual = pick_col_exact(df, cand)
            if actual is not None:
                break
        if actual is None:
            missing.append(label)
        else:
            resolved[label] = actual
    if missing:
        die(f"Missing required column(s) in '{sheet_label}': {', '.join(missing)}. Available columns: {list(df.columns)}")
    return resolved

# =====================
# Extractors
# =====================
def extract_mitigations(xlsx_path: Path) -> None:
    excel_or_die(xlsx_path)
    xls = pd.ExcelFile(xlsx_path, engine="openpyxl")
    sheets = xls.sheet_names

    mit_sheet = find_sheet_exact(sheets, SHEETS["mitigations"]["mitigations"])
    tech_sheet = find_sheet_exact(sheets, SHEETS["mitigations"]["techniques_addressed"])
    missing = [name for name, found in (
        (SHEETS["mitigations"]["mitigations"], mit_sheet),
        (SHEETS["mitigations"]["techniques_addressed"], tech_sheet),
    ) if found is None]
    if missing:
        die(f"Missing required sheet(s) in {xlsx_path.name}: {', '.join(missing)}. Available sheets: {sheets}")

    dfs = pd.read_excel(xlsx_path, sheet_name=[mit_sheet, tech_sheet], engine="openpyxl")
    mit_df = dfs[mit_sheet]
    tech_df = dfs[tech_sheet]

    # Mitigations -> id, name
    cols = require_cols(
        mit_df, mit_sheet,
        [
            ("mitigations.ID", ["ID"]),
            ("mitigations.name", ["name"]),
        ]
    )
    mit_out = mit_df[[cols["mitigations.ID"], cols["mitigations.name"]]].copy()
    mit_out.columns = ["id", "name"]
    mit_out.to_csv(OUT["mitigations"], index=False)

    # Techniques addressed -> source_id, target_id, source_name
    cols = require_cols(
        tech_df, tech_sheet,
        [
            ("techniques.source ID", ["source ID", "source id"]),
            ("techniques.target ID", ["target ID", "target id"]),
            ("techniques.source name", ["source name"]),
        ]
    )
    tech_out = tech_df[[
        cols["techniques.source ID"],
        cols["techniques.target ID"],
        cols["techniques.source name"],
    ]].copy()
    tech_out.columns = ["source_id", "target_id", "source_name"]
    tech_out.to_csv(OUT["mitigation_technique"], index=False)

def extract_tactics(xlsx_path: Path) -> None:
    excel_or_die(xlsx_path)
    xls = pd.ExcelFile(xlsx_path, engine="openpyxl")
    sheets = xls.sheet_names

    sheet = find_sheet_exact(sheets, SHEETS["tactics"]["tactics"])
    if sheet is None:
        die(f"Sheet named '{SHEETS['tactics']['tactics']}' not found in {xlsx_path.name}. Available sheets: {sheets}")

    df = pd.read_excel(xlsx_path, sheet_name=sheet, engine="openpyxl")
    cols = require_cols(
        df, sheet,
        [
            ("ID", ["ID"]),
            ("name", ["name"]),
            ("description", ["description"]),
        ]
    )
    out = df[[cols["ID"], cols["name"], cols["description"]]].copy()
    out.columns = ["id", "name", "description"]
    out.to_csv(OUT["tactics"], index=False)

def extract_techniques_and_targets(xlsx_path: Path) -> None:
    excel_or_die(xlsx_path)
    xls = pd.ExcelFile(xlsx_path, engine="openpyxl")
    sheets = xls.sheet_names

    tech_sheet = find_sheet_exact(sheets, SHEETS["techniques"]["techniques"])
    targets_sheet = find_sheet_exact(sheets, SHEETS["techniques"]["targeted_assets"])
    missing = [name for name, found in (
        (SHEETS["techniques"]["techniques"], tech_sheet),
        (SHEETS["techniques"]["targeted_assets"], targets_sheet),
    ) if found is None]
    if missing:
        die(f"Missing required sheet(s) in {xlsx_path.name}: {', '.join(missing)}. Available sheets: {sheets}")

    dfs = pd.read_excel(xlsx_path, sheet_name=[tech_sheet, targets_sheet], engine="openpyxl")
    tech_df = dfs[tech_sheet]
    targets_df = dfs[targets_sheet]

    # Techniques -> id, name, description
    cols = require_cols(
        tech_df, tech_sheet,
        [
            ("techniques.ID", ["ID"]),
            ("techniques.name", ["name"]),
            ("techniques.description", ["description"]),
        ]
    )
    tech_out = tech_df[[
        cols["techniques.ID"],
        cols["techniques.name"],
        cols["techniques.description"],
    ]].copy()
    tech_out.columns = ["id", "name", "description"]
    tech_out.to_csv(OUT["techniques"], index=False)

    # Targeted assets -> source_id, source_name, target_id, target_name
    cols = require_cols(
        targets_df, targets_sheet,
        [
            ("targeted_assets.source ID", ["source ID", "source id"]),
            ("targeted_assets.source name", ["source name"]),
            ("targeted_assets.target ID", ["target ID", "target id"]),
            ("targeted_assets.target name", ["target name"]),
        ]
    )
    targets_out = targets_df[[
        cols["targeted_assets.source ID"],
        cols["targeted_assets.source name"],
        cols["targeted_assets.target ID"],
        cols["targeted_assets.target name"],
    ]].copy()
    targets_out.columns = ["source_id", "source_name", "target_id", "target_name"]
    targets_out.to_csv(OUT["technique_assets"], index=False)

def extract_groups_bundle(xlsx_path: Path) -> None:
    excel_or_die(xlsx_path)
    xls = pd.ExcelFile(xlsx_path, engine="openpyxl")
    sheets = xls.sheet_names

    g_sheet = find_sheet_exact(sheets, SHEETS["groups"]["groups"])
    a_sheet = find_sheet_exact(sheets, SHEETS["groups"]["associated_software"])
    t_sheet = find_sheet_exact(sheets, SHEETS["groups"]["techniques_used"])
    missing = [name for name, found in (
        (SHEETS["groups"]["groups"], g_sheet),
        (SHEETS["groups"]["associated_software"], a_sheet),
        (SHEETS["groups"]["techniques_used"], t_sheet),
    ) if found is None]
    if missing:
        die(f"Missing required sheet(s) in {xlsx_path.name}: {', '.join(missing)}. Available: {sheets}")

    dfs = pd.read_excel(xlsx_path, sheet_name=[g_sheet, a_sheet, t_sheet], engine="openpyxl")
    groups_df = dfs[g_sheet]
    assoc_df = dfs[a_sheet]
    tech_df  = dfs[t_sheet]

    # groups: id, name, description
    cols = require_cols(
        groups_df, g_sheet,
        [
            ("groups.ID", ["ID"]),
            ("groups.name", ["name"]),
            ("groups.description", ["description"]),
        ]
    )
    groups_out = groups_df[[
        cols["groups.ID"],
        cols["groups.name"],
        cols["groups.description"],
    ]].copy()
    groups_out.columns = ["id", "name", "description"]
    groups_out.to_csv(OUT["groups"], index=False)

    # associated software: source_id, target_id, target_name
    cols = require_cols(
        assoc_df, a_sheet,
        [
            ("associated_software.source_id", ["source ID", "source id"]),
            ("associated_software.target_id", ["target ID", "target id"]),
            ("associated_software.target_name", ["target name"]),
        ]
    )
    assoc_out = assoc_df[[
        cols["associated_software.source_id"],
        cols["associated_software.target_id"],
        cols["associated_software.target_name"],
    ]].copy()
    assoc_out.columns = ["source_id", "target_id", "target_name"]
    assoc_out.to_csv(OUT["associated_software"], index=False)

    # techniques used: source_id, target_id
    cols = require_cols(
        tech_df, t_sheet,
        [
            ("techniques_used.source_id", ["source ID", "source id"]),
            ("techniques_used.target_id", ["target ID", "target id"]),
        ]
    )
    tech_used_out = tech_df[[
        cols["techniques_used.source_id"],
        cols["techniques_used.target_id"],
    ]].copy()
    tech_used_out.columns = ["source_id", "target_id"]
    tech_used_out.to_csv(OUT["group_technique"], index=False)

def extract_assets(xlsx_path: Path) -> None:
    excel_or_die(xlsx_path)
    xls = pd.ExcelFile(xlsx_path, engine="openpyxl")
    sheets = xls.sheet_names

    sheet = find_sheet_exact(sheets, SHEETS["assets"]["assets"])
    if sheet is None:
        die(f"Sheet named '{SHEETS['assets']['assets']}' not found in {xlsx_path.name}. Available sheets: {sheets}")

    df = pd.read_excel(xlsx_path, sheet_name=sheet, engine="openpyxl")

    # Required columns
    # platform: allow "platforms" or "platform"
    # Build a small helper to resolve one-of columns.
    def resolve_one_of(df, candidates):
        for cand in candidates:
            col = pick_col_exact(df, cand)
            if col is not None:
                return col
        return None

    id_col = pick_col_exact(df, "ID")
    name_col = pick_col_exact(df, "name")
    desc_col = pick_col_exact(df, "description")
    plat_col = resolve_one_of(df, ["platforms", "platform"])

    missing = []
    if id_col is None:   missing.append("assets.ID")
    if name_col is None: missing.append("assets.name")
    if desc_col is None: missing.append("assets.description")
    if plat_col is None: missing.append("assets.platforms/platform")

    if missing:
        die(f"Missing required column(s) in '{sheet}': {', '.join(missing)}. Available columns: {list(df.columns)}")

    # flags for platforms
    plat_series = df[plat_col].fillna("").astype(str)
    linux_flags    = plat_series.str.contains(r"\blinux\b", flags=re.I, regex=True)
    windows_flags  = plat_series.str.contains(r"\b(win|windows|ms-windows|microsoft)\b", flags=re.I, regex=True)
    embedded_flags = plat_series.str.contains(r"\bembed|embedded\b", flags=re.I, regex=True)

    out = pd.DataFrame({
        "id": df[id_col].astype(str).fillna(""),
        "name": df[name_col].astype(str).fillna(""),
        "description": df[desc_col].astype(str).fillna(""),
        "linux": linux_flags.fillna(False).astype(bool),
        "windows": windows_flags.fillna(False).astype(bool),
        "embedded": embedded_flags.fillna(False).astype(bool),
    })
    out.to_csv(OUT["assets"], index=False)

# =====================
# Main
# =====================
def main():
    extract_mitigations(MITIGATIONS_XLSX)
    extract_tactics(TACTICS_XLSX)
    extract_techniques_and_targets(TECHNIQUES_XLSX)
    extract_groups_bundle(GROUPS_XLSX)
    extract_assets(ASSETS_XLSX)
    # silent success

if __name__ == "__main__":
    main()
